# Week 4: data, time split, card sequences, XGBoost baseline

**Gate:** baseline recall at 1% FPR reported, and the sequence dataset saved.

**Kaggle (recommended for this week):** New Notebook → *Add Input* → competition **ieee-fraud-detection**
(join the competition and accept its rules first). Settings: Internet **on**, accelerator **None**
(XGBoost on CPU is fine here; save GPU hours for week 5). Kaggle CPU notebooks have ~30 GB RAM.

**Colab:** upload your `kaggle.json` when asked; the data downloads to `/content/ieee`.

In [ ]:
import os, sys, subprocess
REPO = "https://github.com/rc-mathew/deep-learning-fraud-detection.git"
if not os.path.exists("deep-learning-fraud-detection"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
else:
    subprocess.run(["git", "-C", "deep-learning-fraud-detection", "pull", "-q"], check=True)
sys.path.insert(0, "deep-learning-fraud-detection/src")

In [ ]:
ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    DATA_DIR = "/kaggle/input/ieee-fraud-detection"
    OUT_DIR = "/kaggle/working/artifacts"
else:  # Colab
    from google.colab import files, drive
    drive.mount("/content/drive")
    if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
        files.upload()  # pick kaggle.json
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        os.replace("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    DATA_DIR = "/content/ieee"
    if not os.path.exists(f"{DATA_DIR}/train_transaction.csv"):
        subprocess.run(["kaggle", "competitions", "download", "-c", "ieee-fraud-detection", "-p", DATA_DIR], check=True)
        subprocess.run(["unzip", "-q", "-o", f"{DATA_DIR}/ieee-fraud-detection.zip", "-d", DATA_DIR], check=True)
    OUT_DIR = "/content/drive/MyDrive/fraud-dl/artifacts"
print(DATA_DIR, "->", OUT_DIR)

## 1. Load, split by time, build velocity features and card sequences

In [ ]:
import json, time
from fraud_dl.data import load_raw
from fraud_dl.prepare import prepare

t0 = time.time()
raw = load_raw(DATA_DIR)
print(f"{len(raw):,} rows x {raw.shape[1]} cols, {raw.memory_usage().sum() / 1e9:.1f} GB, {time.time() - t0:.0f}s")
report = prepare(raw, OUT_DIR)
del raw
print(json.dumps(report, indent=1))

Read the report before training:
- **share_cards_seen_in_train** for test: how much of the test set has card history the model has seen.
- **fraud_rate_after_card_had_fraud** vs **fraud_rate_overall**: IEEE-CIS labels later transactions of a
  defrauded account as fraud too. If the first number is far above the second, card-history models
  (the week 5 transformer) can look better than they would in production. Note it in the README.

## 2. XGBoost baseline (the one row XGBoost gets)

In [ ]:
import pandas as pd
from pathlib import Path
from fraud_dl.baseline import append_results_row, top_features, train_baseline

art = Path(OUT_DIR)
df = pd.read_parquet(art / "transactions.parquet")
model, results = train_baseline(df, use_gpu=False)
results["top_features_by_gain"] = top_features(model)
(art / "models").mkdir(exist_ok=True)
model.save_model(art / "models" / "xgb_baseline.json")
(art / "xgb_baseline_results.json").write_text(json.dumps(results, indent=1))
append_results_row(results, art / "results.csv")
print(json.dumps({k: results[k] for k in ("best_iteration", "train_seconds", "val", "test")}, indent=1))
pd.read_csv(art / "results.csv")

In [ ]:
pd.Series(results["top_features_by_gain"]).head(15)

## 3. Check the sequence dataset loads as windows

In [ ]:
import numpy as np
from fraud_dl.sequences import window_indices
seq = pd.read_parquet(art / "sequences" / "tokens.parquet")
meta = json.loads((art / "sequences" / "meta.json").read_text())
w = window_indices(np.arange(len(seq)), seq["group_start"].to_numpy(), meta["seq_len"])
lengths = (w >= 0).sum(1)
print(f"{len(seq):,} sequences; history length median {np.median(lengths):.0f}, "
      f"share with full {meta['seq_len']}-txn window {np.mean(lengths == meta['seq_len']):.1%}")
print(pd.Series(lengths).describe())

**Keep the outputs for week 5.** Kaggle: *Save Version → Save & Run All*, then in the week 5 notebook
*Add Input → Notebook Output* of this notebook. Colab: they are already in Drive.
Paste `results.csv` and the data report back into chat.